In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

In [0]:
abt = spark.table("gold.sf_salaries.abt_salario_regressao")
display(abt)

In [0]:
abt = abt.toPandas()

In [0]:
abt.head()

## Análise Exploratória de Dados

In [0]:
abt.info()

In [0]:
abt.isnull().sum()

In [0]:
abt['target_salario'].describe()

In [0]:

plt.figure(figsize=(10, 5))

sns.histplot(data=abt, x="target_salario", bins=100, kde=True, color="#d9534f")

plt.title("Visão Geral da Target", fontsize=14)
plt.xlabel("Variação Salarial ($)", fontsize=12)
plt.ylabel("Quantidade de Funcionários", fontsize=12)
plt.grid(axis='y', alpha=0.3)

plt.show()

In [0]:
df_zoom = abt[(abt['target_salario'] >= -20000) & (abt['target_salario'] <= 20000)]

plt.figure(figsize=(10, 5))

sns.histplot(data=df_zoom, x="target_salario", bins=60, kde=True, color="#0275d8")

plt.title("Zoom na Grande Massa da Target", fontsize=14)
plt.xlabel("Variação Salarial ($)", fontsize=12)
plt.ylabel("Quantidade de Funcionários", fontsize=12)
plt.grid(axis='y', alpha=0.3)

plt.show()

In [0]:
# Maiores médias de aumento salarial por cargo
abt.groupby("cargo").agg({
    "target_salario": "mean",
    "total_funcionarios_cargo": "max"
}).sort_values("target_salario", ascending=False).head(10).reset_index()



In [0]:
# Correlação estatística entre as suas features e a Target

# Seleciona apenas as colunas numéricas disponíveis
colunas_disponiveis = [col for col in [
    "porcentagem_salario_extra", "porcentagem_outros_pagamentos", 
    "porcentagem_beneficios", "porcentagem_salario_base", 
    "desvio_medio_salarial", "target_salario"
] if col in abt.columns]

# Calcula a matriz de correlação de Pearson
matriz_corr = abt[colunas_disponiveis].corr()
print("Matriz de Correlação")
print(matriz_corr["target_salario"])


In [0]:
# Análise temporal das safras
abt.groupby("dt_ref").agg({
    "target_salario": "mean",
    "id_funcionario": "count"
}).sort_values("dt_ref").reset_index()


In [0]:
# Configura o Boxplot para comparar o porcentagem_salario_base entre os anos
plt.figure(figsize=(10, 6))
sns.boxplot(data=abt, x='dt_ref', y='porcentagem_salario_base', palette='Set2')
plt.title("Distribuição do porcentagem_salario_base por Ano de Referência (Safra)", fontsize=14)
plt.xlabel("Ano da Safra (dt_ref)", fontsize=12)
plt.ylabel("porcentagem_salario_base ($)", fontsize=12)
plt.grid(axis='y', alpha=0.3)
plt.show()

In [0]:
# Filtra apenas a safra polêmica de 2012
df_2012 = abt[abt['dt_ref'] == '2012'].copy()

# Criar faixas de variação salarial
def categorizar_mudanca(valor):
    if valor < -100: return 'Teve Redução Salarial'
    elif valor > 100: return 'Teve Aumento Salarial'
    else: return 'Salário Estagnado (Próximo a zero)'

df_2012['categoria_mudanca'] = df_2012['target_salario'].apply(categorizar_mudanca)

# Calcula a contagem e a proporção de funcionários em cada categoria
resumo_2012 = df_2012['categoria_mudanca'].value_counts(normalize=True) * 100
print("--- Comportamento dos Funcionários na Safra 2012 ---")
print(resumo_2012.round(2).astype(str) + '%')


In [0]:

plt.figure(figsize=(10, 6))
# Remove os outliers extremos apenas do visual para conseguirmos enxergar a tendência
df_scatter_filtered = abt[(abt['target_salario'] >= -50000) & (abt['target_salario'] <= 50000)]

sns.regplot(data=df_scatter_filtered, x='desvio_medio_salarial', y='target_salario', 
            scatter_kws={'alpha':0.3, 'color':'gray'}, line_kws={'color':'red'})

plt.title("Desvio Médio Salarial vs Variação Salarial Futura", fontsize=14)
plt.xlabel("Desvio Médio Salarial do Cargo", fontsize=12)
plt.ylabel("Variação Salarial no Ano Seguinte ($)", fontsize=12)
plt.show()


## Definição das Hipóteses para a Safra 2012
**Hipótese Nula (H0):** A variação salarial média em 2012 é estatisticamente igual a 0 (Estagnação absoluta).
**Hipótese Alternativa (H1):** A variação salarial média em 2012 é diferente de 0 (Houve um efeito real de aumento ou redução).

In [0]:
# Remove valores nulos caso existam
dados_2012 = df_2012['target_salario'].dropna()

# 2. Executa o teste t comparando a média amostral contra o valor hipotético 0
t_stat, p_valor = stats.ttest_1samp(dados_2012, popmean=0.0)

print("=== RESULTADO DO TESTE ESTATÍSTICO (SAFRA 2012) ===")
print(f"Média Real Amostral: ${dados_2012.mean():.4f}")
print(f"Estatística t: {t_stat:.4f}")
print(f"p-valor: {p_valor:.4f}")

# 3. Regra de Decisão baseada no nível de significância padrão de 5% (0.05)
alfa = 0.05
if p_valor < alfa:
    print("\n[RESULTADO]: REJEITAMOS a Hipótese Nula (H0).")
    print("Conclusão: Embora o valor médio de $5.21 seja muito baixo, ele é estatisticamente diferente de zero absoluto.")
else:
    print("\n[RESULTADO]: NÃO REJEITAMOS a Hipótese Nula (H0).")
    print("Conclusão: A variação de $5.21 é fruto do acaso. Podemos afirmar cientificamente que o salário ficou estagnado em zero.")
